# Phase 27: Cohort thứ ba (Parmigiani et al. 2022), chạy theo protocol `paper/cohort3_protocol.md` v1.0

Protocol được viết ngày 21/10/2026, sau kiểm kê Phase 26 và **trước** khi chạy mô hình nào.

**Runtime: GPU + High-RAM** (A100 nếu có). Mỗi bệnh nhân có khoảng 150 kênh EEG, tức khoảng 7.000 feature, nên readout nặng hơn ScaleOLDNEW nhiều. Ước tính vài giờ cho cả 31 bệnh nhân. Mọi bước **tiếp tục được** nếu bị ngắt: chạy lại đúng cell đó.

1. **27A:** tải lại dữ liệu vào ổ local (khoảng 10–20 phút; ổ local bị xóa sau mỗi phiên).
2. **27B:** chuyển sang định dạng của pipeline, lưu lên Drive (`data/parm27`, khoảng 1 GB) kèm execution log. Sau bước này 27A không cần chạy lại nữa.
3. **27C:** chép encoder short (đã huấn luyện trên ds004752 cho Phase 17).
4. **27D:** readout: learned, fixed và descriptive 20 %.
5. **27E:** report H1–H3.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
RUNS_DIR = f'{ROOT}/runs'
ENC17    = f'{ROOT}/runs/p17_encoders'
DATA27   = f'{ROOT}/data/parm27'
RUNS27   = f'{ROOT}/runs/p27'
DST      = '/content/parm'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
print('ổ local trống:', round(shutil.disk_usage('/content').free / 1e9), 'GB;', 'encoder short có sẵn:', os.path.exists(f'{ENC17}/short/enc_final.pt'))
!apt-get -qq install -y p7zip-full zip unzip > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%writefile {CODE_DIR}/s2i/parm26.py
"""
Phase 26: download and inventory of the third cohort (Parmigiani et al. 2022, Brain Stimulation 15:664-675;
OSF wsgzp, CC BY-NC-SA 4.0). No model is fitted here: this phase only establishes what the data contain, so that the
protocol for this cohort can be written before any model is applied.

Dataset facts known before this phase (read from the archive's central directory and small files, 21 Oct 2026):
  - BIDS derivative "epochs" only (no continuous recordings): per run, EEG and SEEG epochs of 1001 samples at 1000 Hz
    (-0.3 ... +0.7 s around each single-pulse stimulation), float64, baseline-corrected (mean of -0.3 ... 0 s);
  - EEG: 185 channels, average reference, 0.5-45 Hz (3rd-order Butterworth), ICA for blinks;
    SEEG: bipolar, 0.5-300 Hz;
  - EEG epochs were rejected separately: additional_info/eeg_badtrials.json flags, per run, the SEEG epochs whose
    EEG counterpart was removed (1 = removed), which defines the EEG-SEEG pairing.

  python -m s2i.parm26 download  --dst /content/parm
  python -m s2i.parm26 extract   --dst /content/parm
  python -m s2i.parm26 inventory --dst /content/parm --out <runs>/parm26
"""
import argparse
import collections
import glob
import json
import os
import re
import subprocess
import sys

import numpy as np

PARTS = [("ccepcoreg-bids.z01", 4294967296, "https://osf.io/download/7h5pj/"),
         ("ccepcoreg-bids.z02", 4294967296, "https://osf.io/download/cktnz/"),
         ("ccepcoreg-bids.z03", 4294967296, "https://osf.io/download/9g7kw/"),
         ("ccepcoreg-bids.z04", 4294967296, "https://osf.io/download/d4g9j/"),
         ("ccepcoreg-bids.z05", 4294967296, "https://osf.io/download/ghtb2/"),
         ("ccepcoreg-bids.zip", 2698060289, "https://osf.io/download/agw23/")]
BADTRIALS = ("eeg_badtrials.json", 37419, "https://osf.io/download/4qn3s/")
FS, T0 = 1000, 300                       # sampling rate; index of the stimulation pulse (t = 0) in each epoch
WINDOWS = [(-300, -10), (-10, -2), (-2, 0), (0, 2), (2, 5), (5, 10), (10, 20), (20, 50), (50, 100), (100, 300),
           (300, 701)]                   # ms relative to the pulse


# ----------------------------------------------------------------------------- download / extract
def download(a):
    os.makedirs(a.dst, exist_ok=True)
    for name, size, url in PARTS + [BADTRIALS]:
        f = os.path.join(a.dst, name)
        if os.path.exists(f) and os.path.getsize(f) == size:
            print("ok (present)", name)
            continue
        for attempt in range(5):
            subprocess.run(["curl", "-L", "-sS", "--retry", "5", "-C", "-", "-o", f, url])
            if os.path.exists(f) and os.path.getsize(f) == size:
                break
            print(f"  {name}: {os.path.getsize(f) if os.path.exists(f) else 0} of {size} bytes, retrying")
        if not (os.path.exists(f) and os.path.getsize(f) == size):
            sys.exit(f"download of {name} failed")
        print("ok", name, size)


def extract(a):
    root = os.path.join(a.dst, "ccepcoreg")
    if os.path.isdir(os.path.join(root, "derivatives", "epochs")):
        print("already extracted:", root)
        return
    z = os.path.join(a.dst, "ccepcoreg-bids.zip")
    r = subprocess.run(["7z", "x", "-y", f"-o{a.dst}", z], capture_output=True, text=True)
    if r.returncode != 0 or not os.path.isdir(os.path.join(root, "derivatives", "epochs")):
        print("7z could not read the split archive; joining it with zip -s 0 instead\n", r.stdout[-2000:], r.stderr[-2000:])
        joined = os.path.join(a.dst, "joined.zip")
        subprocess.run(["zip", "-s", "0", z, "--out", joined], check=True)
        for name, _, _ in PARTS:
            os.remove(os.path.join(a.dst, name))
        subprocess.run(["unzip", "-q", "-o", joined, "-d", a.dst], check=True)
        os.remove(joined)
    n = len(glob.glob(os.path.join(root, "derivatives", "epochs", "sub-*", "*", "*_epochs.npy")))
    print("extracted", n, "epoch files to", root)
    if a.delete_parts:
        for name, _, _ in PARTS:
            f = os.path.join(a.dst, name)
            if os.path.exists(f):
                os.remove(f)


# ----------------------------------------------------------------------------- reading
def read_tsv(f):
    lines = open(f, encoding="utf-8").read().splitlines()
    head = lines[0].split("\t")
    return [dict(zip(head, l.split("\t"))) for l in lines[1:] if l.strip()]


def runs_of(root):
    """[(subject, run, eeg_npy, ieeg_npy)] for every run that has both modalities."""
    out = []
    for f in sorted(glob.glob(os.path.join(root, "derivatives", "epochs", "sub-*", "ieeg", "*_epochs.npy"))):
        m = re.search(r"(sub-\d+)_task-ccepcoreg_(run-\d+)_epochs\.npy$", f)
        e = f.replace(os.sep + "ieeg" + os.sep, os.sep + "eeg" + os.sep)
        out.append((m.group(1), m.group(2), e if os.path.exists(e) else None, f))
    return out


def stim_pair(desc):
    """'Stimulation of I'6-7 1ma ...' -> ("I'", 6, 7)."""
    m = re.search(r"Stimulation of\s+(\S+?)(\d+)-(\d+)\s", desc)
    return (m.group(1), int(m.group(2)), int(m.group(3))) if m else None


def near_stim(ch, sp):
    """SEEG bipolar channel 'I'5-6' shares a contact with the stimulated pair, or is the pair itself."""
    m = re.match(r"(\S+?)(\d+)-(\d+)$", ch)
    if not (m and sp) or m.group(1) != sp[0]:
        return False
    return bool({int(m.group(2)), int(m.group(3))} & {sp[1], sp[2]})


def win_rms(x, w):
    a, b = T0 + w[0], T0 + w[1]
    return np.sqrt(np.mean(x[..., a:b] ** 2, axis=-1))          # (epochs, channels)


# ----------------------------------------------------------------------------- inventory
def inventory(a):
    root = os.path.join(a.dst, "ccepcoreg")
    bad = json.load(open(os.path.join(a.dst, BADTRIALS[0])))
    os.makedirs(a.out, exist_ok=True)
    rows, prof = [], collections.defaultdict(list)
    for s, r, fe, fi in runs_of(root):
        base = fi[:-len("_epochs.npy")]
        js = json.load(open(base + "_epochs.json"))
        ch_i = read_tsv(base + "_channels.tsv")
        ch_e = read_tsv(fe[:-len("_epochs.npy")] + "_channels.tsv") if fe else []
        Xi = np.load(fi, mmap_mode="r")
        Xe = np.load(fe, mmap_mode="r") if fe else None
        flags = bad.get(s, {}).get(r)
        row = dict(sub=s, run=r, desc=js.get("Description", ""), shape_ieeg=list(Xi.shape),
                   shape_eeg=list(Xe.shape) if Xe is not None else None, n_flags=len(flags) if flags else None,
                   n_kept_flags=(len(flags) - int(sum(flags))) if flags else None,
                   ch_ieeg_listed=len(ch_i), ch_eeg_listed=len(ch_e),
                   bad_ieeg=sum(c.get("status") == "bad" for c in ch_i), bad_eeg=sum(c.get("status") == "bad" for c in ch_e),
                   units=sorted({c.get("units") for c in ch_i + ch_e}),
                   cut_ieeg=sorted({(c.get("low_cutoff"), c.get("high_cutoff")) for c in ch_i}),
                   cut_eeg=sorted({(c.get("low_cutoff"), c.get("high_cutoff")) for c in ch_e}),
                   ref_eeg=sorted({c.get("reference") for c in ch_e}))
        row["paired"] = (Xe is not None and flags is not None and len(flags) == Xi.shape[0]
                         and row["n_kept_flags"] == Xe.shape[0] and Xi.shape[2] == Xe.shape[2] == 1001
                         and len(ch_i) == Xi.shape[1] and len(ch_e) == Xe.shape[1])
        sp = stim_pair(row["desc"])
        names = [c["name"] for c in ch_i]
        row["stim_pair"] = f"{sp[0]}{sp[1]}-{sp[2]}" if sp else None
        row["stim_in_seeg"] = bool(sp) and f"{sp[0]}{sp[1]}-{sp[2]}" in names
        row["n_seeg_near_stim"] = sum(near_stim(n, sp) for n in names)
        m = re.search(r"(\d+(?:\.\d+)?)hz", row["desc"])
        row["stim_hz"] = float(m.group(1)) if m else None
        if row["paired"]:
            keep = np.array(flags) == 0
            xi = np.asarray(Xi[keep], dtype=np.float64) * 1e6          # microvolts
            xe = np.asarray(Xe, dtype=np.float64) * 1e6
            good_i = np.array([c.get("status") != "bad" for c in ch_i])
            good_e = np.array([c.get("status") != "bad" for c in ch_e])
            near = np.array([near_stim(n, sp) for n in names])
            groups = {"eeg": xe[:, good_e], "seeg_far": xi[:, good_i & ~near], "seeg_near": xi[:, good_i & near]}
            row["baseline_mean_over_sd"] = {k: float(np.median(np.abs(v[..., :T0].mean(-1)) /
                                                               (v[..., :T0].std(-1) + 1e-12))) if v.shape[1] else None
                                            for k, v in groups.items()}
            row["baseline_sd_uv"] = {k: float(np.median(v[..., 10:290].std(-1))) if v.shape[1] else None
                                     for k, v in groups.items()}
            for k, v in groups.items():
                if not v.shape[1]:
                    continue
                b = win_rms(v, WINDOWS[0])
                prof[k].append([float(np.median(win_rms(v, w) / (b + 1e-12))) for w in WINDOWS])
            # share of post-stimulus variance (10-700 ms) that is time-locked: variance of the run average / total
            for k in ("eeg", "seeg_far"):
                v = groups[k][..., T0 + 10:]
                if v.shape[1] and v.shape[0] > 1:
                    ev = v.mean(0).var(-1)
                    tot = v.var(-1).mean(0)
                    row[f"evoked_share_{k}"] = float(np.median(ev / (tot + 1e-12)))
            # timing of the largest EEG deflection around the pulse (artefact alignment)
            pk = np.abs(groups["eeg"][..., T0 - 10:T0 + 20]).mean(0).mean(0).argmax() - 10
            row["eeg_peak_ms"] = int(pk)
            row["n_pairs"] = int(keep.sum())
        rows.append(row)
        print(s, r, "paired" if row["paired"] else "NOT PAIRED", row["shape_ieeg"], row["shape_eeg"], row["stim_pair"],
              row.get("n_pairs"), flush=True)
    json.dump(rows, open(os.path.join(a.out, "runs.json"), "w"), indent=1)

    # ---- coordinates and participants
    coords = {}
    for s in sorted({r["sub"] for r in rows}):
        d = os.path.join(root, "derivatives", "epochs", s)
        mni = glob.glob(os.path.join(d, "ieeg", "*space-MNI152*_electrodes.tsv"))
        eeg_el = glob.glob(os.path.join(d, "eeg", "*_electrodes.tsv"))
        coords[s] = dict(ieeg_mni=len(read_tsv(mni[0])) if mni else 0, eeg_electrodes=len(read_tsv(eeg_el[0])) if eeg_el else 0)
    scans = sorted(glob.glob(os.path.join(root, "derivatives", "epochs", "sub-*", "*_scans.tsv")))
    scan_cols = sorted({k for f in scans for k in (read_tsv(f)[0].keys() if read_tsv(f) else [])})

    # ---- report
    L = ["# Phase 26: inventory of the Parmigiani et al. 2022 dataset (no model fitted)", ""]
    P = [r for r in rows if r["paired"]]
    subs = sorted({r["sub"] for r in rows})
    L += [f"Runs with both modalities: {len(rows)}; EEG-SEEG pairing consistent: {len(P)} "
          f"(flags length = SEEG epochs, kept flags = EEG epochs, 1001 samples, channel lists match the arrays). "
          f"Subjects: {len(subs)}.", ""]
    bad_runs = [r for r in rows if not r["paired"]]
    if bad_runs:
        L += ["Runs that could not be paired:", ""] + [f"- {r['sub']} {r['run']}: SEEG {r['shape_ieeg']}, EEG "
                                                        f"{r['shape_eeg']}, flags {r['n_flags']}/{r['n_kept_flags']}"
                                                        for r in bad_runs] + [""]
    L += ["| subject | runs | paired epochs | paired minutes | SEEG ch (min-max) | EEG ch (min-max) | stim Hz | "
          "SEEG MNI rows | EEG electrodes |", "|" + "---|" * 9]
    for s in subs:
        R = [r for r in P if r["sub"] == s]
        if not R:
            L.append(f"| {s} | 0 | – | – | – | – | – | {coords[s]['ieeg_mni']} | {coords[s]['eeg_electrodes']} |")
            continue
        n = sum(r["n_pairs"] for r in R)
        ci = [r["shape_ieeg"][1] - r["bad_ieeg"] for r in R]
        ce = [r["shape_eeg"][1] - r["bad_eeg"] for r in R]
        hz = sorted({r["stim_hz"] for r in R})
        L.append(f"| {s} | {len(R)} | {n} | {n / 60:.1f} | {min(ci)}-{max(ci)} | {min(ce)}-{max(ce)} | {hz} | "
                 f"{coords[s]['ieeg_mni']} | {coords[s]['eeg_electrodes']} |")
    tot = [sum(r["n_pairs"] for r in P if r["sub"] == s) for s in subs]
    L += ["", f"Paired epochs per subject: median {int(np.median(tot))}, range {min(tot)}-{max(tot)}; total {sum(tot)} "
              f"({sum(tot) / 60:.0f} min of 1-s epochs).", ""]
    L += ["**Filters / reference / units** (from channels.tsv): SEEG cut-offs "
          f"{sorted({tuple(x) for r in rows for x in r['cut_ieeg']})}, EEG cut-offs "
          f"{sorted({tuple(x) for r in rows for x in r['cut_eeg']})}, EEG reference {sorted({x for r in rows for x in r['ref_eeg']})}, "
          f"units {sorted({x for r in rows for x in r['units']})}.", ""]
    L += [f"**Stimulated pair among the SEEG channels:** {sum(r['stim_in_seeg'] for r in rows)} of {len(rows)} runs; "
          f"SEEG channels sharing a contact with the stimulated pair per run: median "
          f"{int(np.median([r['n_seeg_near_stim'] for r in rows]))}, max {max(r['n_seeg_near_stim'] for r in rows)}.", ""]
    L += [f"**Run order:** scans.tsv columns {scan_cols} (no acquisition time means the run number is the only order "
          "available).", ""]
    for k in ("eeg", "seeg_far", "seeg_near"):
        bm = [r["baseline_mean_over_sd"][k] for r in P if r["baseline_mean_over_sd"].get(k) is not None]
        sd = [r["baseline_sd_uv"][k] for r in P if r["baseline_sd_uv"].get(k) is not None]
        if bm:
            L.append(f"- {k}: |pre-stimulus mean| / SD median {np.median(bm):.3f} (0 = baseline-corrected); "
                     f"pre-stimulus SD median {np.median(sd):.1f} µV (runs: {len(bm)})")
    L += ["", "**Artefact profile**: median over runs of the RMS in each window relative to the pre-stimulus RMS "
              "(-300 to -10 ms).", "", "| window (ms) | " + " | ".join(prof) + " |", "|" + "---|" * (len(prof) + 1)]
    for j, w in enumerate(WINDOWS):
        L.append(f"| {w[0]} to {w[1]} | " + " | ".join(f"{np.median([p[j] for p in prof[k]]):.2f}" for k in prof) + " |")
    for k in ("eeg", "seeg_far"):
        v = [r[f"evoked_share_{k}"] for r in P if f"evoked_share_{k}" in r]
        if v:
            L.append(f"\nTime-locked share of post-stimulus variance (10-700 ms; variance of the run average / mean "
                     f"single-epoch variance), {k}: median {np.median(v):.3f}, IQR {np.percentile(v, 25):.3f}-"
                     f"{np.percentile(v, 75):.3f}.")
    pk = collections.Counter(r["eeg_peak_ms"] for r in P)
    L.append(f"\nLatency of the largest mean |EEG| deflection between -10 and +20 ms: {dict(sorted(pk.items()))}.")
    open(os.path.join(a.out, "inventory.md"), "w", encoding="utf-8").write("\n".join(L) + "\n")
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("download", "extract", "inventory"))
    ap.add_argument("--dst", required=True, help="local working folder (Colab disk), about 55 GB")
    ap.add_argument("--out", help="inventory: output folder")
    ap.add_argument("--delete_parts", action="store_true", help="extract: delete the archive parts afterwards")
    a = ap.parse_args()
    {"download": download, "extract": extract, "inventory": inventory}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE_DIR}/s2i/parm27.py
"""
Phase 27 - third cohort (Parmigiani et al. 2022; OSF wsgzp, CC BY-NC-SA 4.0), exactly as in
paper/cohort3_protocol.md (v1.0, 21 Oct 2026, written after the Phase 26 inventory and before any model was applied).

Commands
  build   extracted ccepcoreg tree (Phase 26) -> one s2i data file per included patient (format read by
          s2i.readout --strict) + p27_build.md / p27_build.json (execution log)
  runs    copy the final short encoder trained on ds004752 (s2i.shared --final --half 40 -> <enc_root>/short/
          enc_final.pt, the encoder used for ScaleOLDNEW) to <runs>/short/enc_<sub>.pt
  report  H1-H3 (two-sided Wilcoxon, Holm over H1-H3, median direction, patient bootstrap CI) + descriptive analyses
          -> p27_report.md / p27_report.json

Rules (protocol sections in brackets)
  runs [3]        a run is used only if its EEG-SEEG pairing is consistent: len(eeg_badtrials flags) = number of SEEG
                  epochs and the number of zero flags = number of EEG epochs (the k-th zero flag pairs SEEG epoch with
                  EEG epoch k); runs in run-number order (the only order available)
  channels [4]    EEG: status "good" in every used run; CAR over these channels. SEEG: bipolar channels as shared,
                  status "good" in every used run, sharing no contact with the stimulated pair of any used run
  signals [4]     each epoch (-0.3 ... +0.7 s at 1000 Hz) resampled to 200 Hz (scipy resample_poly 1/5), microvolts
  inclusion [3]   >= 100 paired epochs, >= 3 EEG channels, >= 3 valid targets (non-zero variance in calibration and test)
  split [5]       epochs in run order; calibration = first round(0.3 N); one-epoch gap; the rest is the test set
Storage as in s2i.scales17: robust-normalised with the patient's median / MAD of all kept samples, float16, with the
statistics (one copy per run = session); s2i.readout --strict re-normalises with the calibration epochs only.

python -m s2i.parm27 build  --src /content/parm --out DATA27
python -m s2i.parm27 runs   --enc_root ENC17 --data DATA27 --runs RUNS27
python -m s2i.parm27 report --data DATA27 --runs RUNS27
"""
import argparse
import glob
import json
import os
import re
import shutil
import sys
import time

import numpy as np

FS_IN, DOWN, FS = 1000, 5, 200
CAL_FRAC, MIN_EPOCHS, MIN_EEG, MIN_TGT = 0.3, 100, 3, 3


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} | {msg}", flush=True)


def n_cal_of(n, frac=CAL_FRAC):
    return max(3, int(round(n * frac)))


def robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-30, 1.0, mad)


def read_tsv(f):
    lines = open(f, encoding="utf-8").read().splitlines()
    head = lines[0].split("\t")
    return [dict(zip(head, l.split("\t"))) for l in lines[1:] if l.strip()]


def contacts_of(ch):
    m = re.match(r"(\S+?)(\d+)-(\d+)$", ch)
    return (m.group(1), {int(m.group(2)), int(m.group(3))}) if m else (None, set())


def stim_pair(desc):
    m = re.search(r"Stimulation of\s+(\S+?)(\d+)-(\d+)\s", desc)
    return (m.group(1), {int(m.group(2)), int(m.group(3))}) if m else (None, set())


# ----------------------------------------------------------------------------- one patient
def patient_runs(root, bad, sid):
    """-> [(run, base_ieeg, base_eeg, flags, description)] with consistent pairing, and the list of skipped runs."""
    d = os.path.join(root, "derivatives", "epochs", sid)
    use, skip = [], []
    for fi in sorted(glob.glob(os.path.join(d, "ieeg", f"{sid}_task-ccepcoreg_run-*_epochs.npy")),
                     key=lambda p: int(re.search(r"run-(\d+)", p).group(1))):
        run = re.search(r"(run-\d+)", fi).group(1)
        fe = fi.replace(os.sep + "ieeg" + os.sep, os.sep + "eeg" + os.sep)
        flags = bad.get(sid, {}).get(run)
        if not os.path.exists(fe) or flags is None:
            skip.append((run, "missing EEG file or flags"))
            continue
        ni, ne = np.load(fi, mmap_mode="r").shape[0], np.load(fe, mmap_mode="r").shape[0]
        if len(flags) != ni or len(flags) - int(sum(flags)) != ne:
            skip.append((run, f"pairing inconsistent (SEEG {ni}, EEG {ne}, flags {len(flags)}/{len(flags) - int(sum(flags))})"))
            continue
        desc = json.load(open(fi[:-len("_epochs.npy")] + "_epochs.json")).get("Description", "")
        use.append((run, fi[:-len("_epochs.npy")], fe[:-len("_epochs.npy")], np.asarray(flags), desc))
    return use, skip


def build_patient(root, bad, sid):
    from scipy.signal import resample_poly
    info = {"subject": sid, "excluded": None}
    use, skip = patient_runs(root, bad, sid)
    info["runs_used"], info["runs_skipped"] = [u[0] for u in use], skip
    if not use:
        info["excluded"] = "no run with consistent pairing"
        return info, None
    chi = [read_tsv(u[1] + "_channels.tsv") for u in use]
    che = [read_tsv(u[2] + "_channels.tsv") for u in use]
    names_i, names_e = [c["name"] for c in chi[0]], [c["name"] for c in che[0]]
    assert all([c["name"] for c in x] == names_i for x in chi) and all([c["name"] for c in x] == names_e for x in che), \
        f"{sid}: channel lists differ between runs"
    good_e = [n for j, n in enumerate(names_e) if all(x[j].get("status") == "good" for x in che)]
    stims = [stim_pair(u[4]) for u in use]
    near = set()
    for n in names_i:
        sh, cs = contacts_of(n)
        if any(sh == s and cs & c for s, c in stims):
            near.add(n)
    good_i = [n for j, n in enumerate(names_i) if all(x[j].get("status") == "good" for x in chi) and n not in near]
    info.update(n_eeg_listed=len(names_e), n_eeg=len(good_e), n_seeg_listed=len(names_i),
                n_seeg_near_stim=len(near), n_seeg=len(good_i),
                stim_pairs=[f"{s}{'-'.join(map(str, sorted(c)))}" for s, c in stims])
    ie, ii = [names_e.index(n) for n in good_e], [names_i.index(n) for n in good_i]
    E, Y, ses, desc = [], [], [], []
    for run, bi, be, flags, d in use:
        xi = np.load(bi + "_epochs.npy", mmap_mode="r")
        xe = np.load(be + "_epochs.npy", mmap_mode="r")
        keep = np.where(flags == 0)[0]
        for k, j in enumerate(keep):                          # k-th EEG epoch <-> j-th SEEG epoch
            e = resample_poly(np.asarray(xe[k][ie], np.float64) * 1e6, 1, DOWN, axis=-1)
            y = resample_poly(np.asarray(xi[j][ii], np.float64) * 1e6, 1, DOWN, axis=-1)
            E.append(e - e.mean(0, keepdims=True))            # CAR over the kept EEG channels
            Y.append(y)
            ses.append(run)
        desc.append({"run": run, "description": d, "n_pairs": int(len(keep))})
    N = len(E)
    nc = n_cal_of(N)
    info.update(n_epochs=N, n_cal=nc, n_test=max(N - nc - 1, 0), calib_minutes=nc * E[0].shape[1] / FS / 60,
                test_minutes=max(N - nc - 1, 0) * E[0].shape[1] / FS / 60, samples_per_epoch=int(E[0].shape[1]),
                runs=desc, calib_runs=sorted(set(ses[:nc]), key=ses.index), test_runs=sorted(set(ses[nc + 1:]), key=ses.index))
    valid = [k for k in range(len(good_i))
             if np.var(np.concatenate([y[k] for y in Y[:nc]])) > 0 and np.var(np.concatenate([y[k] for y in Y[nc + 1:]])) > 0]
    info["n_valid_targets"] = len(valid)
    if N < MIN_EPOCHS:
        info["excluded"] = f"{N} paired epochs (< {MIN_EPOCHS})"
    elif len(good_e) < MIN_EEG:
        info["excluded"] = f"{len(good_e)} EEG channels (< {MIN_EEG})"
    elif len(valid) < MIN_TGT:
        info["excluded"] = f"{len(valid)} valid targets (< {MIN_TGT})"
    return info, (E, [y[valid] for y in Y], ses, good_e, [good_i[k] for k in valid])


def write_patient(a, root, sid, data):
    import torch
    E, Y, ses, eeg_names, bip_names = data
    mni = {}
    f = glob.glob(os.path.join(root, "derivatives", "epochs", sid, "ieeg", "*space-MNI152*_electrodes.tsv"))
    if f:
        for r_ in read_tsv(f[0]):
            try:
                mni[r_["name"]] = [float(r_["x"]), float(r_["y"]), float(r_["z"])]
            except (KeyError, ValueError):
                pass
    pos = {}
    f = glob.glob(os.path.join(root, "derivatives", "epochs", sid, "eeg", "*_electrodes.tsv"))
    if f:
        for r_ in read_tsv(f[0]):
            try:
                pos[r_["name"]] = [float(r_["x"]), float(r_["y"]), float(r_["z"])]
            except (KeyError, ValueError):
                pass
    runs = list(dict.fromkeys(ses))
    d = {"subject": sid, "dataset": "Parmigiani2022", "fs": FS, "level": "primary", "eeg_names": eeg_names,
         "bip_names": bip_names, "eeg_pos": [pos.get(n, [np.nan] * 3) for n in eeg_names],
         "bip_mni": torch.tensor([mni.get(n, [np.nan] * 3) for n in bip_names], dtype=torch.float32),
         "epoch_t0_sample": 60, "norm_clip_z": float("inf"), "norm_stats": {}}
    bounds, c = [], 0
    for e in E:
        bounds.append((c, c + e.shape[1]))
        c += e.shape[1]
    d["trial_bounds"] = torch.tensor(bounds, dtype=torch.long)
    d["trial_session"] = list(ses)
    n_inf = 0
    for k, X in (("eeg_car", np.concatenate(E, 1)), ("ieeg_bip", np.concatenate(Y, 1))):
        med, mad = robust(X)
        z = (X - med) / mad
        n_inf += int((np.abs(z) > 6e4).sum())
        d[k] = torch.from_numpy(np.clip(z, -6e4, 6e4).astype(np.float16))
        st = {"med": med[:, 0].tolist(), "mad": mad[:, 0].tolist()}
        d["norm_stats"][k] = {r_: st for r_ in runs}             # one copy per run (= session in s2i.readout)
    os.makedirs(a.out, exist_ok=True)
    fp = os.path.join(a.out, f"{sid}.pt")
    torch.save(d, fp + ".tmp")
    os.replace(fp + ".tmp", fp)
    torch.save({k: d[k] for k in ("subject", "norm_stats", "norm_clip_z", "trial_bounds", "trial_session")},
               os.path.join(a.out, f"{sid}_normstats.pt"))
    return n_inf


def build(a):
    root = os.path.join(a.src, "ccepcoreg")
    bad = json.load(open(os.path.join(a.src, "eeg_badtrials.json")))
    os.makedirs(a.out, exist_ok=True)
    jp = os.path.join(a.out, "p27_build.json")
    res = json.load(open(jp)) if os.path.exists(jp) else {"patients": {}}
    res["rules"] = {"fs_in": FS_IN, "fs": FS, "cal_frac": CAL_FRAC, "min_epochs": MIN_EPOCHS, "min_eeg": MIN_EEG,
                    "min_targets": MIN_TGT}
    subs = sorted(os.path.basename(p) for p in glob.glob(os.path.join(root, "derivatives", "epochs", "sub-*")))
    for sid in subs:
        if a.patients and sid not in a.patients:
            continue
        if sid in res["patients"] and not a.overwrite:
            log(f"[{sid}] already built")
            continue
        t0 = time.time()
        info, data = build_patient(root, bad, sid)
        if info["excluded"] is None:
            info["float16_overflow_samples"] = write_patient(a, root, sid, data)
        res["patients"][sid] = info
        json.dump(res, open(jp + ".tmp", "w"), indent=1, default=str)
        os.replace(jp + ".tmp", jp)
        log(f"[{sid}] runs {len(info['runs_used'])} (skipped {len(info['runs_skipped'])}), {info.get('n_epochs', 0)} "
            f"epochs (calib {info.get('n_cal', 0)}, test {info.get('n_test', 0)}), EEG {info.get('n_eeg', 0)}, SEEG "
            f"{info.get('n_valid_targets', 0)}, excluded: {info['excluded']} | {time.time() - t0:.0f}s")
    write_build_md(a, res)


def write_build_md(a, res):
    P = res["patients"]
    L = ["# Phase 27 execution log: data (cohort-3 protocol v1.0, Sections 3-5)", "", f"- rules: {res['rules']}", "",
         "| patient | runs used | runs skipped | epochs | calib | test | calib min | test min | EEG ch (listed) | "
         "SEEG targets (listed, near stim) | calib runs | excluded |", "|" + "---|" * 12]
    for sid, i in sorted(P.items()):
        L.append(f"| {sid} | {len(i['runs_used'])} | {len(i['runs_skipped'])} | {i.get('n_epochs', 0)} | "
                 f"{i.get('n_cal', '')} | {i.get('n_test', '')} | {i.get('calib_minutes', 0):.2f} | "
                 f"{i.get('test_minutes', 0):.1f} | {i.get('n_eeg', '')} ({i.get('n_eeg_listed', '')}) | "
                 f"{i.get('n_valid_targets', '')} ({i.get('n_seeg_listed', '')}, {i.get('n_seeg_near_stim', '')}) | "
                 f"{', '.join(i.get('calib_runs', []))} | {i['excluded'] or ''} |")
    inc = [s for s, i in P.items() if i["excluded"] is None]
    L += ["", f"included: {len(inc)} patients: {', '.join(sorted(inc))}", "", "Skipped runs:", ""]
    L += [f"- {sid}: " + "; ".join(f"{r} ({w})" for r, w in i["runs_skipped"]) for sid, i in sorted(P.items())
          if i["runs_skipped"]]
    out = "\n".join(L)
    open(os.path.join(a.out, "p27_build.md"), "w", encoding="utf-8").write(out + "\n")
    print(out)


# ----------------------------------------------------------------------------- runs
def runs(a):
    subs = sorted(os.path.basename(p)[:-3] for p in glob.glob(os.path.join(a.data, "sub-*.pt"))
                  if not p.endswith("_normstats.pt"))
    src = os.path.join(a.enc_root, "short", "enc_final.pt")
    assert os.path.exists(src), f"missing final short encoder {src}"
    od = os.path.join(a.runs, "short")
    os.makedirs(od, exist_ok=True)
    for s in subs:
        dst = os.path.join(od, f"enc_{s}.pt")
        if not os.path.exists(dst):
            shutil.copy(src, dst)
    log(f"short encoder copied for {len(subs)} patients -> {od}")


# ----------------------------------------------------------------------------- report
def _load(run, name):
    out = {}
    for f in glob.glob(os.path.join(run, name, "readout_sub-*.json")):
        d = json.load(open(f))
        out[d["subject"]] = d
    return out


def _ex(D, m, k="r"):
    return {s: r["explicit"][m]["mean"][k] for s, r in D.items() if m in r.get("explicit", {})
            and k in r["explicit"][m]["mean"]}


def compare(a_, b_, rng):
    from scipy.stats import wilcoxon
    s = sorted(set(a_) & set(b_))
    x, y = np.array([a_[q] for q in s]), np.array([b_[q] for q in s])
    d = x - y
    if len(d) == 0:
        return None
    bt = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)])
    try:
        p = float(wilcoxon(x, y, alternative="two-sided").pvalue)
    except ValueError:
        p = float("nan")
    return {"n": len(d), "mean_a": float(x.mean()), "mean_b": float(y.mean()), "median_diff": float(np.median(d)),
            "mean_diff": float(d.mean()), "ci": [float(np.percentile(bt, 2.5)), float(np.percentile(bt, 97.5))],
            "wins": int((d > 0).sum()), "p": p, "per_patient": dict(zip(s, d.round(5).tolist()))}


def holm(ps):
    order = np.argsort(ps)
    adj, prev = [0.0] * len(ps), 0.0
    for rank, j in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[j]))
        adj[j] = prev
    return adj


def report(a):
    rng = np.random.default_rng(0)
    R = os.path.join(a.runs, "short")
    B = json.load(open(os.path.join(a.data, "p27_build.json")))["patients"]
    learned, fixed = _load(R, "learned_primary"), _load(R, "fixed_primary")
    included = sorted(s for s, i in B.items() if i["excluded"] is None)
    L = ["# Phase 27: third cohort (Parmigiani et al. 2022), cohort-3 protocol v1.0", "",
         f"- included patients: {len(included)}: {', '.join(included)}"]
    for lab, D in (("learned", learned), ("fixed", fixed)):
        if sorted(D) != included:
            L.append(f"- **WARNING**: {lab} results exist for {len(D)} of {len(included)} patients: run incomplete")
    H = [("H1 (system, r)", "Final-short", "Base", compare(_ex(learned, "combo_nosess"), _ex(learned, "base"), rng)),
         ("H2 (learning, r)", "Learned-short", "Fixed-short",
          compare(_ex(learned, "combo_nosess"), _ex(fixed, "combo_nosess"), rng)),
         ("H3 (system, r_induced_ses)", "Final-short", "Base",
          compare(_ex(learned, "combo_nosess", "r_induced_ses"), _ex(learned, "base", "r_induced_ses"), rng))]
    adj = holm([h[3]["p"] for h in H])
    out = {}
    L += ["", "## Pre-specified hypotheses (two-sided Wilcoxon, Holm over H1-H3)", "",
          "| hypothesis | A | B | n | mean A | mean B | median A − B | mean A − B [95 % CI] | A > B | p | p (Holm) | supported |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for (name, la, lb, h), pa in zip(H, adj):
        sup = pa < 0.05 and h["median_diff"] > 0
        h["p_holm"], h["supported"] = pa, bool(sup)
        out[name] = h
        L.append(f"| {name} | {la} | {lb} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['median_diff']:+.4f} | "
                 f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.2e} | "
                 f"{pa:.2e} | {'**yes**' if sup else 'no'} |")
    L += ["", "Per-patient differences (A − B):", ""]
    for name, _, _, h in H:
        L.append(f"- {name}: " + ", ".join(f"{s} {v:+.4f}" for s, v in h["per_patient"].items()))
    L += ["", "## Descriptive analyses (unadjusted p)", "",
          "| comparison | metric | n | mean A | mean B | mean A − B [95 % CI] | median A − B | A > B | p |",
          "|---|---|---|---|---|---|---|---|---|"]
    desc = []

    def row(lab, A, B, metric):
        h = compare(A, B, rng)
        if h is None:
            return
        desc.append({"comparison": lab, "metric": metric, **h})
        L.append(f"| {lab} | {metric} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['mean_diff']:+.4f} "
                 f"[{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['median_diff']:+.4f} | {h['wins']}/{h['n']} | {h['p']:.2e} |")
    for metric in ("r_induced", "r_no50", "r2"):
        row("Final-short − Base", _ex(learned, "combo_nosess", metric), _ex(learned, "base", metric), metric)
    row("Learned-short − Fixed-short", _ex(learned, "combo_nosess", "r_induced_ses"),
        _ex(fixed, "combo_nosess", "r_induced_ses"), "r_induced_ses")
    row("Fixed-short − Base", _ex(fixed, "combo_nosess"), _ex(fixed, "base"), "r")
    row("Learned encoder alone − Base", _ex(learned, "enc"), _ex(learned, "base"), "r")
    l20 = _load(R, "learned_primary_f20")
    if l20:
        row("Final-short, 20 % calibration − Base, 20 %", _ex(l20, "combo_nosess"), _ex(l20, "base"), "r")
        row("Final-short, 20 % − Base, 30 %", _ex(l20, "combo_nosess"), _ex(learned, "base"), "r")
    out["descriptive"] = desc
    L += ["", "Mean over patients:", "", "| model | r | r_induced_ses | r_induced | r_no50 | R² | constant predictions |",
          "|---|---|---|---|---|---|---|"]
    for lab, D, m in (("Base", learned, "base"), ("Fixed-short (+ LAG9)", fixed, "combo_nosess"),
                      ("Learned encoder alone", learned, "enc"), ("Final-short (learned + LAG9)", learned, "combo_nosess")):
        v = [_ex(D, m, k) for k in ("r", "r_induced_ses", "r_induced", "r_no50", "r2")]
        nc = sum(r["explicit"][m].get("n_const", 0) for r in D.values() if m in r.get("explicit", {}))
        L.append(f"| {lab} | " + " | ".join(f"{np.mean(list(x.values())):.4f}" if x else "" for x in v) + f" | {nc} |")
    L += ["", "Per patient (r):", "", "| patient | epochs | calib min | EEG | targets | Base | Fixed-short | Final-short |",
          "|---|---|---|---|---|---|---|---|"]
    fb, ff, fl = _ex(learned, "base"), _ex(fixed, "combo_nosess"), _ex(learned, "combo_nosess")
    for s in included:
        i = B[s]
        L.append(f"| {s} | {i['n_epochs']} | {i['calib_minutes']:.2f} | {i['n_eeg']} | {i['n_valid_targets']} | "
                 f"{fb.get(s, float('nan')):.3f} | {ff.get(s, float('nan')):.3f} | {fl.get(s, float('nan')):.3f} |")
    md = "\n".join(L)
    print(md)
    open(os.path.join(a.runs, "p27_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(a.runs, "p27_report.json"), "w"), indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("build", "runs", "report"))
    ap.add_argument("--src", help="build: Phase-26 folder with ccepcoreg/ and eeg_badtrials.json")
    ap.add_argument("--out", help="build: output data folder")
    ap.add_argument("--data", help="runs / report: data folder")
    ap.add_argument("--enc_root", help="runs: folder with short/enc_final.pt (Phase 17 encoders)")
    ap.add_argument("--runs", help="runs / report: run root")
    ap.add_argument("--patients", nargs="*", default=None)
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"build": build, "runs": runs, "report": report}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE_DIR}/s2i/readout.py
"""
Phase 4 - readout improvements on the SAVED shared encoders (CPU, no retraining).

Phase 3 analysis of the shared encoder (15 subjects, 20 % calibration):
  (1) training traded high frequencies for low ones: the untrained Morlet bank (enc0) beats the
      trained encoder in alpha/beta, the trained one wins in delta/theta; two pairs collapsed on 50 Hz;
  (2) half of the features (pairs >= 13 Hz and their envelopes) sit in bands with little predictable
      signal, but one shared ridge penalty treats them like the useful ones;
  (3) at 1-2 min of calibration the readout (C x 40 weights per contact) is the bottleneck.
Every candidate below keeps the encoder fixed and changes only the readout:

  enc         reference = enc_ridge of s2i.shared (ridge, one K-fold alpha)
  base        reference = ridge_kfold_shared on LAG9
  enc_low     only low-frequency encoder features (pairs < 13 Hz, their envelopes, free filters)
  enc_group   high-frequency features down-weighted by a K-fold-chosen factor (two penalties)
  enc_enc0hi  + the UNTRAINED encoder's high-frequency features (alpha/beta), own K-fold weight
  enc_lag9    + LAG9 (raw lags +-200 ms), own K-fold weight
  enc_bands   target split into 5 complementary bands (<4, 4-8, 8-13, 13-30, >30 Hz); one ridge per
              band with its own K-fold penalty (chosen by MSE); predictions summed
  enc_sess    encoder features standardised per recording session (EEG only, label-free,
              transductive: uses the EEG of every trial of the session)
  enc_warp    filters time-scaled by s in {0.9, 1, 1.1} (frequencies x 1/s), s chosen per subject by
              K-fold (individual frequency adaptation)
  enc_bilin   structured readout W_k = a_k b_k^T (channels x features, rank 1-3, spectral profiles b
              orthonormal, ridge penalty on a), rank and penalty by K-fold: (C + F) R weights per contact
  ens_base    enc and base averaged with a K-fold-chosen weight
  ens_seed    enc of this run and of the seed-1 run averaged with a K-fold-chosen weight (if present)
--strict: undo preprocessing's whole-session normalisation and re-normalise EEG and target with the
calibration trials' statistics only; session-standardised models are skipped (combo_nosess = combo
without session standardisation is reported in every mode).
Phase 4b (--only_combo, combinations of the three Phase-4 winners):
  ens_all       ridge readouts of ALL available encoders (main + seed runs) averaged
  ens_all_sess  same, every encoder's features standardised per session
  combo         ens_all_sess mixed with base (LAG9 ridge), weight chosen by K-fold

Exactness: every readout is linear in its features, so K-fold out-of-fold r, the amplitude gain and
test r / R^2 are computed EXACTLY from second-order sufficient statistics (sums of x, xx', y, y^2,
xy') accumulated in one pass over the trials. `enc` therefore reproduces enc_ridge of s2i.shared.
Test trials, calibration fractions and fold blocks are those of phases 0-3.

Usage
-----
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car [--seed_run RUNS_DIR/shared_seed1]
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car --only_combo
       --seed_runs RUNS_DIR/shared_seed1 RUNS_DIR/shared_seed2     (Phase 4b: combinations of the winners)
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import ALPHAS, chrono_split, feat_lag, template, tmpl_at
from s2i.baselines import DEFAULT_LAGS
from s2i.metrics import per_contact_metrics
from s2i.shared import TemporalEncoder, np_feat, r_single_trial, r_without_line
from s2i.within import ALPHAS_WIDE, _blocks

FS = 200
HIGH_HZ = 13.0
Y_CUTS = (4.0, 8.0, 13.0, 30.0)                       # -> 5 complementary target bands
GROUP_SCALES = (1.0, 0.5, 0.25, 0.1, 0.0)             # weight of the high-frequency group
ADD_SCALES = (0.0, 0.25, 0.5, 1.0, 2.0)               # weight of an added feature block
ENS_W = (0.0, 0.25, 0.5, 0.75, 1.0)                   # weight of enc in an ensemble
WARPS = (0.9, 1.1)
BILIN = tuple((r, al) for r in (1, 2, 3) for al in (0.001, 0.01, 0.1, 1.0, 10.0))
LAG9 = feat_lag(DEFAULT_LAGS)
_LP = {c: butter(4, c, btype="low", fs=FS, output="sos") for c in Y_CUTS}
_DEV = None                                           # set in main(); Gram matrices on the GPU if cuda


# ----------------------------------------------------------------------------- statistics
class St:
    """Second-order sufficient statistics of (features x, target y); optional band targets."""

    def __init__(self, D, K, nb=0, Db=0):
        self.n = 0.0
        self.sx, self.sxx = np.zeros(D), np.zeros((D, D))
        self.sy, self.syy, self.sxy = np.zeros(K), np.zeros(K), np.zeros((D, K))
        self.syb, self.syyb, self.sxyb = np.zeros((nb, K)), np.zeros((nb, K)), np.zeros((nb, Db, K))

    def add(self, X, y, yb=None):
        self.n += X.shape[1]
        self.sx += X.sum(1)
        if _DEV is not None and _DEV.type == "cuda":             # large Gram matrices on the GPU
            Xt, yt = torch.from_numpy(X).to(_DEV), torch.from_numpy(np.ascontiguousarray(y)).to(_DEV)
            self.sxx += (Xt @ Xt.T).double().cpu().numpy()
            self.sxy += (Xt @ yt.T).double().cpu().numpy()
        else:
            self.sxx += X @ X.T
            self.sxy += X @ y.T
        self.sy += y.sum(1)
        self.syy += (y * y).sum(1)
        if yb is not None:
            Xb = X[:self.sxyb.shape[1]]
            self.syb += yb.sum(2)
            self.syyb += (yb * yb).sum(2)
            for b in range(len(yb)):
                self.sxyb[b] += Xb @ yb[b].T

    def _new(self):
        return St.__new__(St)

    def __add__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) + getattr(o, k))
        return s

    def __sub__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) - getattr(o, k))
        return s

    def take(self, idx, band=None, transform=None):
        """Sub-block of features `idx` (band b replaces the target), optionally x -> (x - m) / d."""
        s = self._new()
        s.n, sx, sxx = self.n, self.sx[idx], self.sxx[np.ix_(idx, idx)]
        if band is None:
            s.sy, s.syy, sxy = self.sy, self.syy, self.sxy[idx]
        else:
            s.sy, s.syy, sxy = self.syb[band], self.syyb[band], self.sxyb[band][idx]
        if transform is not None:
            m, d = transform
            sxx = (sxx - np.outer(m, sx) - np.outer(sx, m) + s.n * np.outer(m, m)) / np.outer(d, d)
            sxy = (sxy - np.outer(m, s.sy)) / d[:, None]
            sx = (sx - s.n * m) / d
        s.sx, s.sxx, s.sxy = sx, sxx, sxy
        K = len(s.sy)
        s.syb, s.syyb, s.sxyb = np.zeros((0, K)), np.zeros((0, K)), np.zeros((0, len(idx), K))
        return s


def total(sts):
    out = sts[0]
    for s in sts[1:]:
        out = out + s
    return out


# ----------------------------------------------------------------------------- linear readouts
def standardise(tr):
    n = tr.n
    mu, ym = tr.sx / n, tr.sy / n
    G = tr.sxx - n * np.outer(mu, mu)
    sd = np.sqrt(np.clip(np.diag(G) / n, 0, None)) + 1e-8
    Gs = G / np.outer(sd, sd)
    Bs = (tr.sxy - n * np.outer(mu, ym)) / sd[:, None]
    return mu, sd, (Gs + Gs.T) / 2, Bs, ym


def to_raw(Ws, mu, sd, ym):
    A = Ws / sd[:, None]
    return A, ym - mu @ A


def ridge_models(tr, alphas, scale_vec=None):
    """Ridge for each alpha (penalty alpha * mean eigenvalue). scale_vec multiplies standardised features."""
    mu, sd, Gs, Bs, ym = standardise(tr)
    sc = max(np.trace(Gs) / len(Gs), 1e-12)      # penalty scale of the UNSCALED design (= n)
    if scale_vec is not None:
        Gs, Bs = Gs * np.outer(scale_vec, scale_vec), Bs * scale_vec[:, None]
    lam, Q = np.linalg.eigh(Gs)
    lam = np.clip(lam, 0, None)
    QB = Q.T @ Bs
    out = []
    for a in alphas:
        Ws = Q @ (QB / (lam + a * sc)[:, None])
        if scale_vec is not None:
            Ws = Ws * scale_vec[:, None]
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def bilinear_models(tr, C, F, cands, iters=5):
    """
    W_k = a_k b_k^T over (channel, feature) with rank R: b_k (F x R) has orthonormal columns (spectral
    profiles), a_k (C x R) carries the ridge penalty alpha * n, exactly the ridge scale. Alternating
    least squares, initialised by the SVD of a ridge fit.
    """
    mu, sd, Gs, Bs, ym = standardise(tr)
    K, sc = Bs.shape[1], max(np.trace(Gs) / len(Gs), 1e-12)
    G4, B3 = Gs.reshape(C, F, C, F), Bs.reshape(C, F, K)
    lam, Q = np.linalg.eigh(Gs)
    W0 = Q @ ((Q.T @ Bs) / (np.clip(lam, 0, None) + sc)[:, None])
    _, _, Vt = np.linalg.svd(W0.T.reshape(K, C, F), full_matrices=False)
    out = []
    for R, alpha in cands:
        R_ = min(R, Vt.shape[1])
        b = Vt[:, :R_, :].transpose(0, 2, 1)                     # (K, F, R) orthonormal columns

        def solve_a(b):
            # Ga[k] = (I_C x b_k)' G (I_C x b_k): contract g with b (one matmul), then f (batched matmul)
            M = (Gs.reshape(C * F * C, F) @ b.transpose(1, 0, 2).reshape(F, K * R_)).reshape(C, F, C, K, R_)
            Ga = np.matmul(M.transpose(3, 0, 2, 4, 1), b[:, None, None]).transpose(0, 1, 4, 2, 3)
            ca = np.einsum("cfk,kfr->kcr", B3, b).reshape(K, C * R_, 1)
            return np.linalg.solve(Ga.reshape(K, C * R_, C * R_) + alpha * sc * np.eye(C * R_),
                                   ca)[..., 0].reshape(K, C, R_)
        a = solve_a(b)
        for _ in range(iters):
            # Gb[k] = (a_k x I_F)' G (a_k x I_F): contract c with a (one matmul), then d (batched matmul)
            M = (a.transpose(0, 2, 1).reshape(K * R_, C) @ Gs.reshape(C, F * C * F)).reshape(K, R_, F, C, F)
            Gb = np.matmul(M.transpose(0, 1, 2, 4, 3), a[:, None, None]).transpose(0, 2, 1, 3, 4)
            Gb = Gb.reshape(K, F * R_, F * R_)
            cb = np.einsum("cfk,kcr->kfr", B3, a).reshape(K, F * R_, 1)
            b = np.linalg.solve(Gb + 1e-4 * sc * np.eye(F * R_), cb)[..., 0].reshape(K, F, R_)
            b, _ = np.linalg.qr(b)                               # keep only the profile subspace
            a = solve_a(b)
        Ws = np.einsum("kcr,kfr->cfk", a, b).reshape(C * F, K)
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def pred_sums(A, c, te):
    """Sums of prediction p = x'A + c over the samples summarised by `te`: (n, sp, spp, spy, sy, syy)."""
    Ax = A.T @ te.sx
    sp = Ax + te.n * c
    spp = ((te.sxx @ A) * A).sum(0) + 2 * c * Ax + te.n * c ** 2
    spy = (A * te.sxy).sum(0) + c * te.sy
    return np.stack([np.full_like(sp, te.n), sp, spp, spy, te.sy, te.syy])


def r_of(s):
    n, sp, spp, spy, sy, syy = s
    den = np.sqrt(np.clip(n * spp - sp ** 2, 0, None) * np.clip(n * syy - sy ** 2, 0, None))
    return np.where(den > 0, (n * spy - sp * sy) / np.maximum(den, 1e-12), np.nan)


def sse_of(s):
    return s[2] - 2 * s[3] + s[5]


class Fit:
    """K-fold selection over candidates from statistics. fit(train_stats) -> [(A, c)] per candidate."""

    def __init__(self, fit, folds, tot, crit="r"):
        self.fit, self.tot = fit, tot
        self.oof = None
        for tr, te in folds:
            s = np.stack([pred_sums(A, c, te) for A, c in fit(tr)])
            self.oof = s if self.oof is None else self.oof + s
        if crit == "r":
            score = [np.nanmean(r_of(o)) for o in self.oof]
            self.best = int(np.nanargmax(score))
        else:
            self.best = int(np.argmin([np.nansum(sse_of(o)) for o in self.oof]))
        self.A, self.c = fit(tot)[self.best]
        self.fold_models = None

    def fold_model(self, folds):
        if self.fold_models is None:
            self.fold_models = [self.fit(tr)[self.best] for tr, _ in folds]
        return self.fold_models


def calibrated(A, c, oof, ym):
    """Per-contact least-squares gain on the pooled out-of-fold predictions (r unchanged, R^2 honest)."""
    n, sp, spp, spy, sy, _ = oof
    num = spy - ym * sp - ym * sy + n * ym ** 2
    den = spp - 2 * ym * sp + n * ym ** 2
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)
    return A * g, ym + g * (c - ym)


def score(A, c, oof, tot, te, te_sessions=None):
    ym = tot.sy / tot.n
    s = pred_sums(A, c, te)
    Ac, cc = calibrated(A, c, oof, ym)
    sc = pred_sums(Ac, cc, te)
    r2 = 1 - sse_of(sc) / np.maximum(sc[5] - sc[4] ** 2 / sc[0], 1e-12)
    out = {"r": float(np.nanmean(r_of(s))), "r2": float(np.nanmean(r2)), "contacts": [float(x) for x in r_of(s)],
           "cv_contacts": [float(x) for x in r_of(oof)]}          # K-fold r on calibration (reliability map)
    if te_sessions:                                            # calibration longevity: r per test session
        out["sessions"] = {k: float(np.nanmean(r_of(pred_sums(A, c, v)))) for k, v in te_sessions.items()}
    return out


def embed(A, c, idx, D):
    out = np.zeros((D, A.shape[1]))
    out[idx] = A
    return out, c


# ----------------------------------------------------------------------------- features
def load_encoder(path, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    enc = TemporalEncoder(**ck["cfg"]).to(dev)
    enc.load_state_dict(ck["state"])
    return enc.eval()


def warped(enc, s, dev):
    """Same encoder with every kernel time-scaled by s: h_s(t) = h(s t), i.e. frequencies x 1/s."""
    out = TemporalEncoder(**enc.cfg).to(dev)
    w = enc.conv.weight.detach().cpu().numpy()[:, 0]
    H = w.shape[1] // 2
    t = np.arange(-H, H + 1)
    ws = np.stack([np.interp(t / s, t, k, left=0.0, right=0.0) for k in w])
    ws *= np.linalg.norm(w, axis=1, keepdims=True) / (np.linalg.norm(ws, axis=1, keepdims=True) + 1e-12)
    with torch.no_grad():
        out.conv.weight.copy_(torch.tensor(ws, dtype=torch.float32)[:, None])
    return out.eval()


def high_mask(enc, use_init):
    """Per-feature mask (length n_feat): True for quadrature pairs >= HIGH_HZ and their envelopes."""
    cfg = enc.cfg
    P = cfg["n_pairs"]
    if use_init:
        f = np.geomspace(cfg["fmin"], cfg["fmax"], P)
    else:
        w = enc.conv.weight.detach().cpu().numpy()[:, 0]
        spec = np.abs(np.fft.rfft(w, 4096, axis=1)) ** 2
        fr = np.fft.rfftfreq(4096, 1 / FS)
        f = fr[np.argmax(spec[0:2 * P:2] + spec[1:2 * P:2], 1)]
    m = np.zeros(enc.n_feat, bool)
    for p in np.where(f >= HIGH_HZ)[0]:
        m[[2 * p, 2 * p + 1]] = True
        if cfg["envelope"]:
            m[2 * P + cfg["n_free"] + p] = True
    return m, f


def band_targets(y):
    lp = [sosfiltfilt(_LP[c], y, axis=1) for c in Y_CUTS]
    parts = [lp[0]] + [lp[i] - lp[i - 1] for i in range(1, len(lp))] + [y - lp[-1]]
    return np.stack(parts).astype(np.float32)


# ----------------------------------------------------------------------------- per subject
def load_subject(path):
    """-> trials of CAR EEG, trials of bipolar targets, trial sessions, stimulus events (or None)."""
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    b = d["trial_bounds"].numpy()
    ses = list(d.get("trial_session", ["ses-01"] * len(b)))
    ev = d.get("stim_events")
    return ([eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b], ses,
            None if ev is None else np.asarray(ev, int).reshape(-1, 2))


EPOCH = (-100, 500)                                   # -0.5 to +2.5 s around a stimulus marker (samples)


def r_event(P, T, test, events):
    """
    Per-contact r without the stimulus-locked component: each side minus ITS OWN average over the test stimulus
    epochs (-0.5 to +2.5 s around each stimulus marker). events: rows (trial index, onset sample within the trial).
    An epoch is used only if it lies inside one test trial / window; a sample covered by two epochs takes the
    later one. Returns NaN per contact if there is no such epoch.
    """
    pos = {int(w): j for j, w in enumerate(test)}
    ep = sorted((pos[int(w)], int(o)) for w, o in events
                if int(w) in pos and o + EPOCH[0] >= 0 and o + EPOCH[1] <= P[pos[int(w)]].shape[1])
    if not ep:
        return np.full(P[0].shape[0], np.nan)
    tP = np.mean([P[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    tT = np.mean([T[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    Pr, Tr = [p.astype(np.float64) for p in P], [t.astype(np.float64) for t in T]
    for j, o in ep:                                   # ascending: a later epoch overwrites the overlap
        Pr[j][:, o + EPOCH[0]:o + EPOCH[1]] = P[j][:, o + EPOCH[0]:o + EPOCH[1]] - tP
        Tr[j][:, o + EPOCH[0]:o + EPOCH[1]] = T[j][:, o + EPOCH[0]:o + EPOCH[1]] - tT
    return per_contact_metrics(Pr, Tr, FS, bands=False)["r"]


def r_induced_by_session(P, T, test, ses):
    """
    r_induced with the trial average taken per SESSION: each side minus its own average over the test trials of the
    same session (e.g. one stimulation site per run in the third cohort). Same rule as r_single_trial otherwise.
    """
    Pr, Tr = [None] * len(P), [None] * len(T)
    for s in dict.fromkeys(ses[i] for i in test):
        js = [j for j, i in enumerate(test) if ses[i] == s]
        tP, tT = template([P[j] for j in js]), template([T[j] for j in js])
        for j in js:
            Pr[j] = P[j] - tmpl_at(tP, P[j].shape[1])
            Tr[j] = T[j] - tmpl_at(tT, T[j].shape[1])
    return per_contact_metrics(Pr, Tr, FS, bands=False)["r"]


def save_predictions(a, res, preds, Yt, test):
    """pred_<sub>.npz: signals of the best, median and worst contact (final-model test r) for the final model, the
    LAG9 ridge and the recording (float16, all test trials concatenated, with the trial lengths), every contact's r,
    and the magnitude-squared coherence between prediction and recording averaged over contacts."""
    from scipy.signal import coherence
    rr = np.asarray(res["explicit"]["combo_nosess"]["r"], float)
    ok = np.where(np.isfinite(rr))[0]
    order = ok[np.argsort(rr[ok])]
    pick = [int(order[-1]), int(order[len(order) // 2]), int(order[0])]
    T = np.concatenate(Yt, 1).astype(np.float64)
    out = {"pick": np.array(pick), "trial_len": np.array([y.shape[1] for y in Yt]), "test": np.array(test),
           "r_final": rr, "r_base": np.asarray(res["explicit"]["base"]["r"], float),
           "sig_target": T[pick].astype(np.float16)}
    for nm, key in (("combo_nosess", "final"), ("base", "base")):
        P = np.concatenate(preds[nm], 1).astype(np.float64)
        out[f"sig_{key}"] = P[pick].astype(np.float16)
        C = []
        for k in range(T.shape[0]):
            if np.ptp(P[k]) == 0:
                continue                                  # constant prediction: coherence undefined
            fr, c = coherence(P[k], T[k], fs=FS, nperseg=256)
            C.append(c)
        out["freqs"] = fr
        out[f"coh_{key}"] = np.nanmean(C, 0) if C else np.full(129, np.nan)
    np.savez_compressed(os.path.join(a._out, f"pred_{a._sid}.npz"), **out)


def _robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-12, 1.0, mad)


def strict_renorm(E, Y, ses, stats, n_cal):
    """
    Undo the whole-session robust normalisation of preprocessing (x * MAD_s + median_s) and re-normalise every
    trial with the median / MAD of the CALIBRATION trials only (first n_cal), for EEG and target alike:
    no statistic of a test trial is used anywhere (strict, deployment-faithful normalisation).
    """
    def redo(X, key):
        raw = []
        for x, s in zip(X, ses):
            st = stats["norm_stats"][key][s]
            raw.append(x * np.asarray(st["mad"], np.float32)[:, None] + np.asarray(st["med"], np.float32)[:, None])
        med, mad = _robust(np.concatenate(raw[:n_cal], 1))
        return [((r - med) / mad).astype(np.float32) for r in raw]
    return redo(E, "eeg_car"), redo(Y, "ieeg_bip")


def analyze(E, Y, ses, enc, seed_encs, a, dev, log, events=None):
    amp = dev.type == "cuda"
    zoo = not a.only_combo
    n, C, K = len(E), E[0].shape[0], Y[0].shape[0]
    _, test = chrono_split(n, getattr(a, "test_frac", 0.2), gap=1)
    enc0 = TemporalEncoder(**enc.cfg).to(dev).eval()
    F = enc.n_feat
    hi_enc, f_enc = high_mask(enc, use_init=False)
    hi_e0, _ = high_mask(enc0, use_init=True)
    f1, f0 = np_feat(enc, dev, amp), np_feat(enc0, dev, amp)
    fss = [np_feat(se, dev, amp) for se in seed_encs]
    fs1 = fss[0] if fss else None
    fw = {s: np_feat(warped(enc, s, dev), dev, amp) for s in WARPS} if zoo else {}
    e0_rows = np.where(np.tile(hi_e0, C))[0]
    seeds = [f"seed{j}" for j in range(len(fss))]

    # main view = [enc | enc0 high-frequency rows (zoo only) | LAG9 | seed encoders]
    blocks = [("enc", C * F)] + ([("enc0hi", len(e0_rows))] if zoo else []) + [("lag9", C * len(DEFAULT_LAGS))]
    blocks += [(b, C * se.n_feat) for b, se in zip(seeds, seed_encs)]     # encoders may differ (e.g. no_env)
    idx, o = {}, 0
    for name, size in blocks:
        idx[name] = np.arange(o, o + size)
        o += size
    D = o
    hi_rows = np.where(np.tile(hi_enc, C))[0]
    lo_rows = np.where(~np.tile(hi_enc, C))[0]

    def main_x(e):
        x1 = f1(e)
        parts = [x1] + ([f0(e)[e0_rows]] if zoo else []) + [LAG9(e)] + [f(e) for f in fss]
        return np.concatenate(parts).astype(np.float32), x1

    # session statistics of every encoder's features (EEG of every trial, no target)
    enc_fns = {"enc": f1, **dict(zip(seeds, fss))}
    sess_sum = {}
    for i in range(n):
        for b, fn in enc_fns.items():
            x = fn(E[i]).astype(np.float64)
            s = sess_sum.setdefault((b, ses[i]), [0, np.zeros(len(idx[b])), np.zeros(len(idx[b]))])
            s[0] += x.shape[1]
            s[1] += x.sum(1)
            s[2] += (x * x).sum(1)
    sess_tf = {k: (v[1] / v[0], np.sqrt(np.clip(v[2] / v[0] - (v[1] / v[0]) ** 2, 0, None)) + 1e-6)
               for k, v in sess_sum.items()}

    def tf_for(names, session):
        """Per-session standardisation of encoder blocks; identity for other blocks (e.g. LAG9)."""
        m = [sess_tf[(b, session)][0] if (b, session) in sess_tf else np.zeros(len(idx[b])) for b in names]
        d = [sess_tf[(b, session)][1] if (b, session) in sess_tf else np.ones(len(idx[b])) for b in names]
        return np.concatenate(m), np.concatenate(d)

    # test statistics (per session, for the session-standardised readout)
    te_main, te_w = {}, {s: St(C * F, K) for s in fw}
    for i in test:
        X, _ = main_x(E[i])
        te_main.setdefault(ses[i], St(D, K)).add(X, Y[i])
        for s in fw:
            te_w[s].add(fw[s](E[i]).astype(np.float32), Y[i])
    te = total(list(te_main.values()))
    te_enc = te.take(idx["enc"])
    te_sess = total([v.take(idx["enc"], transform=tf_for(["enc"], k)) for k, v in te_main.items()])

    order = list(dict.fromkeys(ses))                     # sessions in recording order
    res = {"n_trials": n, "n_test": len(test), "n_sessions": len({k[1] for k in sess_sum}),
           "session_order": order, "test_trials_per_session": {k: sum(ses[i] == k for i in test) for k in order},
           "n_encoders": 1 + len(fss),
           "pair_peak_hz": [float(x) for x in f_enc], "fracs": {}}
    for frac in a.fracs:
        n_c = max(3, int(round(n * frac)))
        if n_c >= test[0]:
            continue
        t0 = time.time()
        fold_ids = _blocks(n_c, 5)
        cal = {}                                 # (fold, session) -> St (main view, with band targets)
        cal_w = {s: [St(C * F, K) for _ in fold_ids] for s in fw}
        nbt = len(Y_CUTS) + 1 if zoo else 0
        for fi, ids in enumerate(fold_ids):
            for i in ids:
                X, _ = main_x(E[i])
                cal.setdefault((fi, ses[i]), St(D, K, nb=nbt, Db=C * F if zoo else 0)).add(
                    X, Y[i], band_targets(Y[i]) if zoo else None)
                for s in fw:
                    cal_w[s][fi].add(fw[s](E[i]).astype(np.float32), Y[i])
        fold_st = [total([v for (f, _), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
        tot = total(fold_st)

        def folds_of(take):
            return [(take(tot - fs), take(fs)) for fs in fold_st]

        out, extra = {}, {}

        def run(name, take, fit, test_st, crit="r", note=None):
            fl = folds_of(take)
            m = Fit(fit, fl, take(tot), crit)
            out[name] = score(m.A, m.c, m.oof[m.best], take(tot), test_st,
                              {k: take(v) for k, v in te_main.items()})
            if note:
                extra[name] = note(m)
            return m, fl

        t_enc = lambda s: s.take(idx["enc"])                                    # noqa: E731
        m_enc, fl_enc = run("enc", t_enc, lambda tr: ridge_models(tr, ALPHAS_WIDE), te_enc,
                            note=lambda m: ALPHAS_WIDE[m.best])
        t_lag = lambda s: s.take(idx["lag9"])                                   # noqa: E731
        m_base, fl_base = run("base", t_lag, lambda tr: ridge_models(tr, ALPHAS), te.take(idx["lag9"]))
        store = {}                       # model -> (feature rows, calibrated A, c, session-standardised blocks)
        for nm, m_, rows in (("enc", m_enc, idx["enc"]), ("base", m_base, idx["lag9"])):
            ts = tot.take(rows)
            store[nm] = (rows, *calibrated(m_.A, m_.c, m_.oof[m_.best], ts.sy / ts.n), None)

        if zoo:
            lo = idx["enc"][lo_rows]
            run("enc_low", lambda s: s.take(lo), lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(lo))

            gmask = np.zeros(C * F, bool)
            gmask[hi_rows] = True
            cands = [(g, al) for g in GROUP_SCALES for al in ALPHAS_WIDE]

            def fit_group(tr):
                outm = []
                for g in GROUP_SCALES:
                    outm += ridge_models(tr, ALPHAS_WIDE, np.where(gmask, g, 1.0))
                return outm
            run("enc_group", t_enc, fit_group, te_enc, note=lambda m: cands[m.best][0])

            def added(name, block):
                sel = np.concatenate([idx["enc"], idx[block]])
                mask = np.r_[np.zeros(len(idx["enc"]), bool), np.ones(len(idx[block]), bool)]
                cands_ = [(g, al) for g in ADD_SCALES for al in ALPHAS_WIDE]

                def fit(tr):
                    outm = []
                    for g in ADD_SCALES:
                        outm += ridge_models(tr, ALPHAS_WIDE, np.where(mask, g, 1.0))
                    return outm
                run(name, lambda s: s.take(sel), fit, te.take(sel), note=lambda m: cands_[m.best][0])
            added("enc_enc0hi", "enc0hi")
            added("enc_lag9", "lag9")

            # one ridge per target band (penalty by MSE on that band), predictions summed
            nb = len(Y_CUTS) + 1
            bm = [Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), folds_of(lambda s, b=b: s.take(idx["enc"], band=b)),
                      tot.take(idx["enc"], band=b), crit="mse") for b in range(nb)]
            A = sum(m.A for m in bm)
            c = sum(m.c for m in bm)
            fms = [m.fold_model([(tr, None) for tr, _ in folds_of(lambda s, b=b: s.take(idx["enc"], band=b))])
                   for b, m in enumerate(bm)]
            oof = sum(pred_sums(sum(fms[b][fi][0] for b in range(nb)), sum(fms[b][fi][1] for b in range(nb)), te_f)
                      for fi, (_, te_f) in enumerate(fl_enc))
            out["enc_bands"] = score(A, c, oof, t_enc(tot), te_enc)
            extra["enc_bands"] = [ALPHAS_WIDE[m.best] for m in bm]

            # per-session standardisation of the encoder features
            sess_tot = total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()])
            sess_folds = [total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()
                                 if f == fi])
                          for fi in range(len(fold_ids))]
            m = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(sess_tot - fs, fs) for fs in sess_folds], sess_tot)
            out["enc_sess"] = score(m.A, m.c, m.oof[m.best], sess_tot, te_sess)

            # individual frequency adaptation: pick s in {0.9, 1, 1.1} by K-fold r
            best = ("1.0", m_enc, np.nanmean(r_of(m_enc.oof[m_enc.best])), te_enc, t_enc(tot))
            for s in WARPS:
                tw = total(cal_w[s])
                mw = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(tw - fs, fs) for fs in cal_w[s]], tw)
                sw = np.nanmean(r_of(mw.oof[mw.best]))
                if sw > best[2]:
                    best = (f"{s:g}", mw, sw, te_w[s], tw)
            out["enc_warp"] = score(best[1].A, best[1].c, best[1].oof[best[1].best], best[4], best[3])
            extra["enc_warp"] = best[0]

            # structured (bilinear) readout
            if not a.no_bilin:
                run("enc_bilin", t_enc, lambda tr: bilinear_models(tr, C, F, BILIN), te_enc,
                    note=lambda m: list(BILIN[m.best]))

        # ensembles: enc + base, enc + seed-1 (weights chosen by pooled out-of-fold r)
        def ensemble(name, m1, fl1, i1, m2, fl2, i2):
            sel = np.concatenate([i1, i2])
            n1 = len(i1)
            fm1 = m1.fold_model(fl1)
            fm2 = m2.fold_model(fl2)
            t_sel = lambda s: s.take(sel)                                       # noqa: E731
            fl = folds_of(t_sel)
            oofs = []
            for w in ENS_W:
                oofs.append(sum(pred_sums(np.r_[w * fm1[fi][0], (1 - w) * fm2[fi][0]], w * fm1[fi][1] +
                                          (1 - w) * fm2[fi][1], te_f) for fi, (_, te_f) in enumerate(fl)))
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            w = ENS_W[wi]
            A_ = np.r_[w * m1.A, (1 - w) * m2.A]
            out[name] = score(A_, w * m1.c + (1 - w) * m2.c, oofs[wi], t_sel(tot), te.take(sel))
            extra[name] = w
            assert len(A_) == len(sel) and n1 == len(m1.A)
        ensemble("ens_base", m_enc, fl_enc, idx["enc"], m_base, fl_base, idx["lag9"])
        if fss:
            t_seed = lambda s: s.take(idx["seed0"])                             # noqa: E731
            m_seed, fl_seed = run("seed", t_seed, lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(idx["seed0"]))
            ensemble("ens_seed", m_enc, fl_enc, idx["enc"], m_seed, fl_seed, idx["seed0"])

        # combinations of the three Phase-4 winners: several encoders averaged (ens_seed), session
        # standardisation (enc_sess) and mixing with LAG9 ridge (ens_base); exact from statistics
        def mixture(name, encs, sess, fixed_w=None):
            names = encs + ["lag9"]
            sel = np.concatenate([idx[b] for b in names])

            def T(st, session):
                return st.take(sel, transform=tf_for(names, session) if sess else None)
            folds_T = [total([T(v, ss) for (f, ss), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
            tot_T = total(folds_T)
            te_T = total([T(v, ss) for ss, v in te_main.items()])
            comps, o_ = [], 0
            for b in names:
                loc = np.arange(o_, o_ + len(idx[b]))
                o_ += len(idx[b])
                al = ALPHAS if b == "lag9" else ALPHAS_WIDE
                fl = [((tot_T - fs).take(loc), fs.take(loc)) for fs in folds_T]
                m = Fit(lambda tr, al=al: ridge_models(tr, al), fl, tot_T.take(loc))
                comps.append((b, loc, m, m.fold_model(fl)))

            def combine(w, models):
                A_, c_ = np.zeros((len(sel), K)), np.zeros(K)
                for (b, loc, _, _), (Ab, cb) in zip(comps, models):
                    wt = (1 - w) if b == "lag9" else w / len(encs)
                    A_[loc] += wt * Ab
                    c_ += wt * cb
                return A_, c_
            ws = ENS_W if fixed_w is None else (fixed_w,)
            oofs = [sum(pred_sums(*combine(w, [cm[3][fi] for cm in comps]), fs) for fi, fs in enumerate(folds_T))
                    for w in ws]
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            A_, c_ = combine(ws[wi], [(cm[2].A, cm[2].c) for cm in comps])
            out[name] = score(A_, c_, oofs[wi], tot_T, te_T, {ss: T(v, ss) for ss, v in te_main.items()})
            store[name] = (sel, *calibrated(A_, c_, oofs[wi], tot_T.sy / tot_T.n), names if sess else None)
            extra[name] = ws[wi]

        all_encs = ["enc"] + seeds
        if not zoo and not a.strict:
            mixture("enc_sess", ["enc"], sess=True, fixed_w=1.0)
        if seeds:
            mixture("ens_all", all_encs, sess=False, fixed_w=1.0)
            if not a.strict:
                mixture("ens_all_sess", all_encs, sess=True, fixed_w=1.0)
        if not a.strict:                         # session standardisation uses the EEG of every trial (transductive)
            mixture("combo", all_encs, sess=True)
        mixture("combo_nosess", all_encs, sess=False)

        if a.explicit and abs(frac - a.explicit_frac) < 1e-9:
            # explicit test predictions of the final models: band / 50 Hz / single-trial metrics
            res["explicit"] = {}
            Yt = [Y[i] for i in test]
            preds = {}
            for nm in [m for m in ("base", "enc", "combo", "combo_nosess") if m in store]:
                rows, A_, c_, names = store[nm]
                P = []
                for i in test:
                    X = main_x(E[i])[0][rows].astype(np.float64)
                    if names:
                        mm, dd = tf_for(names, ses[i])
                        X = (X - mm[:, None]) / dd[:, None]
                    P.append((X.T @ A_ + c_).T.astype(np.float32))
                met = per_contact_metrics(P, Yt, FS, bands=True)
                met["r_no50"], met["r_induced"] = r_without_line(P, Yt), r_single_trial(P, Yt)
                if getattr(a, "induced_by_session", False):
                    met["r_induced_ses"] = r_induced_by_session(P, Yt, test, ses)
                if events is not None:
                    met["r_event"] = r_event(P, Yt, test, events)
                preds[nm] = P
                Pc = np.concatenate(P, 1)
                res["explicit"][nm] = {"frac": frac, "mean": {k: float(np.nanmean(v)) for k, v in met.items()},
                                       "r": [float(x) for x in met["r"]],
                                       "r_induced": [float(x) for x in met["r_induced"]],
                                       "n_const": int((Pc.max(1) == Pc.min(1)).sum())}   # constant -> r = 0
            if getattr(a, "save_pred", False) and "combo_nosess" in preds and "base" in preds:
                save_predictions(a, res, preds, Yt, test)
            log("    explicit: " + " ".join(f"{k} r={v['mean']['r']:.4f} (stats {out[k]['r']:.4f}) "
                                          f"r_ind={v['mean']['r_induced']:.4f}"
                                          for k, v in res["explicit"].items()))
        mins = sum(E[i].shape[1] for i in range(n_c)) / FS / 60
        res["fracs"][f"{frac:g}"] = {"n_calib": n_c, "minutes": mins, "models": out, "choice": extra,
                                     "calib_sessions": sorted({ses[i] for i in range(n_c)}, key=order.index)}
        log(f"    f={frac:g} ({mins:.1f} min, {time.time() - t0:.0f}s) " +
            " ".join(f"{k}={v['r']:.3f}" for k, v in out.items()))
    return res


# ----------------------------------------------------------------------------- summary
def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon
    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r["fracs"][f] for r in results if f in r["fracs"]]
        models = [m for m in rows[0]["models"] if all(m in r["models"] for r in rows)]
        M = {m: np.array([r["models"][m]["r"] for r in rows]) for m in models}
        R2 = {m: np.array([r["models"][m]["r2"] for r in rows]) for m in models}
        mins = np.mean([r["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs enc | wins vs enc | p vs enc | vs base | R² (calibrated) | K-fold choices |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d = M[m] - M["enc"]
            try:
                p = float(wilcoxon(M[m], M["enc"]).pvalue) if m != "enc" and np.any(d != 0) else float("nan")
            except ValueError:
                p = float("nan")
            ch = [r["choice"].get(m) for r in rows if m in r["choice"]]
            ch = [tuple(x) if isinstance(x, list) else x for x in ch]
            if ch and m != "enc_bands":
                vals, cnt = np.unique([str(x) for x in ch], return_counts=True)
                chs = ", ".join(f"{v}×{c}" for v, c in zip(vals, cnt))
            else:
                chs = ""
            summ[f][m] = {"mean": float(M[m].mean()), "diff_enc": float(d.mean()), "wins_enc": int((d > 0).sum()),
                          "p_enc": p, "diff_base": float((M[m] - M["base"]).mean()), "r2": float(R2[m].mean())}
            lines.append(f"| {m} | {M[m].mean():.4f} | {d.mean():+.4f} | {(d > 0).sum()}/{len(d)} | {p:.4f} | "
                         f"{(M[m] - M['base']).mean():+.4f} | {R2[m].mean():.3f} | {chs} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "readout.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "readout.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="output dir of s2i.shared (enc_<sub>.pt)")
    ap.add_argument("--seed_run", default=None, help="second run for the seed ensemble (enc_<sub>.pt)")
    ap.add_argument("--seed_runs", nargs="*", default=[], help="further runs (encoders) for the ensemble")
    ap.add_argument("--explicit", action="store_true",
                    help="also predict the test trials explicitly with base / enc / combo (all metrics)")
    ap.add_argument("--explicit_frac", type=float, default=0.2)
    ap.add_argument("--strict", action="store_true",
                    help="re-normalise EEG and target with calibration-trial statistics only (needs <sub>_normstats.pt "
                         "from preprocess --stats_only); no session standardisation; requires a single --fracs value")
    ap.add_argument("--stats_dir", default=None, help="folder with <sub>_normstats.pt (default: --data)")
    ap.add_argument("--untrained", action="store_true",
                    help="replace every encoder by its Morlet initialisation (fixed filter bank, no meta-learning)")
    ap.add_argument("--only_combo", action="store_true",
                    help="skip the Phase-4 variants already rejected; compute enc, base, ensembles and combos")
    ap.add_argument("--out_name", default=None, help="output sub-folder (default: readout / readout_combo)")
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--test_frac", type=float, default=0.2,
                    help="the test trials follow the first round(test_frac * n) trials and a one-trial gap (fixed for "
                         "every calibration fraction; fractions above it are skipped)")
    ap.add_argument("--save_pred", action="store_true",
                    help="explicit: save pred_<sub>.npz (three contacts' test signals and coherence spectra)")
    ap.add_argument("--induced_by_session", action="store_true",
                    help="explicit: also r_induced_ses (trial average per session; third cohort, one site per run)")
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--no_bilin", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    if a.strict:
        assert len(a.fracs) == 1, "--strict needs exactly one calibration fraction (the statistics depend on it)"
    out = os.path.join(a.run, a.out_name or ("readout_strict" if a.strict else
                                             "readout_combo" if a.only_combo else "readout"))
    os.makedirs(out, exist_ok=True)
    log_f = open(os.path.join(out, "readout.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    torch.set_grad_enabled(False)
    global _DEV
    _DEV = dev
    seed_runs = ([a.seed_run] if a.seed_run else []) + list(a.seed_runs)
    log(f"config: {vars(a)}")
    results = []
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        if a.subjects and sid not in a.subjects:
            continue
        fp = os.path.join(out, f"readout_{sid}.json")
        sps = [sp for sp in (os.path.join(r_, f"enc_{sid}.pt") for r_ in seed_runs) if os.path.exists(sp)]
        if not os.path.exists(os.path.join(a.data, f"{sid}.pt")):
            log(f"[{sid}] no data file in {a.data} (excluded at this level), skip")
            continue
        if os.path.exists(fp) and not a.overwrite:
            done = json.load(open(fp))
            # resume: keep a finished subject only if it used the same number of encoders as now
            if done.get("n_encoders", 1 + len(sps)) == 1 + len(sps):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
            log(f"[{sid}] done with {done['n_encoders']} encoders, now {1 + len(sps)}: recomputing")
        t0 = time.time()
        enc = load_encoder(ep, dev)
        seed_encs = [load_encoder(sp, dev) for sp in sps]
        if a.untrained:
            enc = TemporalEncoder(**enc.cfg).to(dev).eval()
            seed_encs = [TemporalEncoder(**se.cfg).to(dev).eval() for se in seed_encs]
        E, Y, ses, events = load_subject(os.path.join(a.data, f"{sid}.pt"))
        if a.strict:
            st = torch.load(os.path.join(a.stats_dir or a.data, f"{sid}_normstats.pt"), weights_only=False)
            assert len(st["trial_bounds"]) == len(E) and list(st["trial_session"]) == ses, "stats / data mismatch"
            E, Y = strict_renorm(E, Y, ses, st, max(3, int(round(len(E) * a.fracs[0]))))
        log(f"[{sid}] {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} targets, "
            f"{len(set(ses))} sessions, extra encoders: {len(seed_encs)}")
        a._sid, a._out = sid, out
        r = analyze(E, Y, ses, enc, seed_encs, a, dev, log, events)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        summarize(results, out, log)


if __name__ == "__main__":
    main()


In [ ]:
#@title 27A. Tải + giải nén vào ổ local (bỏ qua nếu 27B đã xong cho mọi bệnh nhân)
!python -m s2i.parm26 download --dst {DST}
!python -m s2i.parm26 extract --dst {DST} --delete_parts

In [ ]:
#@title 27B. Chuyển dữ liệu + execution log (khoảng 20-40 phút)
!python -m s2i.parm27 build --src {DST} --out {DATA27}

In [ ]:
#@title 27C. Encoder short cho từng bệnh nhân
!python -m s2i.parm27 runs --enc_root {ENC17} --data {DATA27} --runs {RUNS27}

In [ ]:
#@title 27D. Readout (tiếp tục được; chạy lại cell này nếu bị ngắt)
R = f'{RUNS27}/short'
COMMON = f'--data {DATA27} --run {R} --only_combo --strict --test_frac 0.3 --explicit --induced_by_session --no_bilin'
!python -m s2i.readout {COMMON} --fracs 0.3 --explicit_frac 0.3 --out_name learned_primary 2>&1 | grep -E "\] |explicit|Error|error"
!python -m s2i.readout {COMMON} --fracs 0.3 --explicit_frac 0.3 --untrained --out_name fixed_primary 2>&1 | grep -E "\] |explicit|Error|error"
!python -m s2i.readout {COMMON} --fracs 0.2 --explicit_frac 0.2 --out_name learned_primary_f20 2>&1 | grep -E "\] |explicit|Error|error"

In [ ]:
#@title 27E. Report H1-H3
!python -m s2i.parm27 report --data {DATA27} --runs {RUNS27}

Gửi tôi output của **27B** (bảng execution log, từ dòng `# Phase 27 execution log`) và **27E** (toàn bộ).